In [2]:
import numpy as np
import pandas as pd
from scipy import constants
import glob 
import matlab.engine
import matplotlib.pyplot as plt
from multiprocessing import Pool

In [3]:
def svd_beamformer_analog(H):
    ant_tx, ant_rx = np.shape(H)
    u, s, wr_opt = np.linalg.svd(H)
    u, s, wt_opt = np.linalg.svd(np.transpose(H))
    
    wr_quant_angle = -np.transpose(np.around(np.angle(wr_opt)/(np.pi/2))*(np.pi/2))
    wt_quant_angle = -np.transpose(np.around(np.angle(wt_opt)/(np.pi/2))*(np.pi/2))
    
    wr_quant = np.exp(1j*wr_quant_angle)
    wt_quant = np.exp(1j*wt_quant_angle)
    
    rss = []
    beam_idx = []
    for i in range(ant_tx):
        for j in range(ant_rx):
            signal = np.abs(np.matmul(wt_quant[:,i],np.matmul(H,wr_quant[:,j])))**2
            rss.append(10*np.log10(signal*1000))
            beam_idx.append([i,j])
    
    idx = np.argmax(np.array(rss))
    tx_idx = beam_idx[idx][0]
    rx_idx = beam_idx[idx][1]
            
    wr_quant_out = wr_quant[:,rx_idx]
    wt_quant_out = wt_quant[:,tx_idx] 
    
    return wt_quant_out, wr_quant_out

In [4]:
eng = matlab.engine.start_matlab()

In [5]:
azi_files = glob.glob('./output/az*')

In [6]:
tx = {'freq':11.325e9,
      'light_speed':constants.c,
      'power': 4.89e6,
      'num_ant': 1,
      'rf_chain':1,
      'ant_spacing':constants.c/11.325e9/2}

rx = {'freq':11.325e9,
      'light_speed':constants.c,
      'power': 1,
      'num_ant': 36,
      'rf_chain':1,
      'ant_spacing':constants.c/11.325e9/2}

In [7]:
noise = -80

In [10]:
def kernel_function(azi_file, lock = 1):
    print("Now Processing " + azi_file)
    alt_file = azi_file.replace('az','alt')
    dist_file = azi_file.replace('az','distance')
    
    azi_data = pd.read_csv(azi_file)
    alt_data = pd.read_csv(alt_file)
    dist_data = pd.read_csv(dist_file)
    
    satellites = list(azi_data.columns)[2:]
    
    rss_ori = np.ones((len(azi_data),len(satellites)))*np.nan
    rss_beamformed = np.ones((len(azi_data),len(satellites)))*np.nan
    rate_ori = np.ones((len(azi_data),len(satellites)))*np.nan
    rate_beamformed = np.ones((len(azi_data),len(satellites)))*np.nan
    for sat_idx, sat in enumerate(satellites):
        txang_azi = azi_data[sat].values
        index = np.where(txang_azi!=0)
        
        txang_azi = txang_azi[txang_azi!=0]-np.pi
        txang_ele = alt_data[sat].values
        txang_ele = txang_ele[txang_ele!=0]
        txang = matlab.double(np.array([txang_azi,txang_ele]).tolist())
       
        rxang_azi = azi_data[sat].values
        rxang_azi = rxang_azi[rxang_azi!=0]-np.pi
        rxang_ele = alt_data[sat].values
        rxang_ele = rxang_ele[rxang_ele!=0]
        rxang = matlab.double(np.array([rxang_azi,rxang_ele]).tolist())
        
        dist = np.array(dist_data[sat].values)*1000
        dist = matlab.double(dist[dist!=0].tolist())

        # print(np.shape(rxang))
        # print(np.shape(txang))
        # print(np.shape(dist))
        
        H = np.array(eng.get_channel(matlab.double(tx["num_ant"]), matlab.double(rx["num_ant"]), 
                            matlab.double(tx["rf_chain"]), matlab.double(rx["rf_chain"]),
                           matlab.double(tx["freq"]), txang, rxang, dist))
        
        rss_ori_cur = []
        rss_beamformed_cur = []
        rate_ori_cur = []
        rate_beamformed_cur = []
        for i in range(len(H)):
            wt, wr = np.squeeze(svd_beamformer_analog(H[i,:,:]))
            
            rss_ori_sat = 10*np.log10(np.abs(np.sum(H[i,:,:]))**2*tx['power']/tx['num_ant']*1000)
            rss_beamformed_sat = np.squeeze(10*np.log10(np.abs(np.matmul(np.expand_dims(wt, axis = 0), np.matmul(H[i,:,:], np.expand_dims(wr, axis = 1))))**2*tx['power']/tx['num_ant']*1000))
            
            rss_ori_cur.append(rss_ori_sat)
            rss_beamformed_cur.append(rss_beamformed_sat)
            
            rate_ori_cur.append(np.log2(1 + 10**((rss_ori_sat - noise)/10)))
            rate_beamformed_cur.append(np.log2(1 + 10**((rss_beamformed_sat - noise)/10)))
        
        rss_ori[index,sat_idx] = rss_ori_cur
        rss_beamformed[index,sat_idx] = rss_beamformed_cur
        rate_ori[index, sat_idx] = rate_ori_cur
        rate_beamformed[index, sat_idx] = rate_beamformed_cur
        
        if np.mod(sat_idx+1, 10) == 0 or sat_idx+1 == len(satellites):
            print("Processed " + alt_file + " " + str(sat_idx+1) + "/" + str(len(satellites)) + " satellite")
    
    fout = azi_file.replace('az','rss_ori')
    df_out = pd.DataFrame(rss_ori.tolist(), columns = satellites)
    df_out.to_csv(fout)

    fout = azi_file.replace('az','rss_beamformed')
    df_out = pd.DataFrame(rss_beamformed.tolist(), columns = satellites)
    df_out.to_csv(fout)
    
    fout = azi_file.replace('az','rate_ori')
    df_out = pd.DataFrame(rate_ori.tolist(), columns = satellites)
    df_out.to_csv(fout)
    
    fout = azi_file.replace('az','rate_beamformed')
    df_out = pd.DataFrame(rate_beamformed.tolist(), columns = satellites)
    df_out.to_csv(fout)

In [11]:
for azi_file in azi_files:
    kernel_function(azi_file, 2)

Now Processing ./output/az_Boston_2022-9-21-18-00-00.csv
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 10/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 20/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 30/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 40/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 50/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 60/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 70/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 80/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 90/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 100/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 110/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 120/147 satellite
Processed ./output/alt_Boston_2022-9-21-18-00-00.csv 130/147 satellite
Processed ./output/alt_Boston